# 📊 Credit Risk Dataset - Exploratory Data Analysis (EDA) & Data Understanding

## 🎯 Project Objective
This notebook performs a comprehensive Exploratory Data Analysis (EDA) on the **Credit Risk Dataset**.
The main goals are:
- Understand customer demographic profiles and loan characteristics.
- Identify data quality issues (missing values, duplicates, outliers, anomalies).
- Analyze risk drivers associated with loan default (`loan_status`).
- Formulate recommendations for data transformation, feature engineering, and credit risk modeling.


In [ ]:
# 1. Import Essential Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Plotting configurations
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")


## 📥 2. Data Ingestion & Initial Inspection
Load the Credit Risk dataset from the `../dataset/` directory and perform initial dimension & structure verification.


In [ ]:
# Load dataset
file_path = "../dataset/Credit Risk Dataset.xlsx"
df = pd.read_excel(file_path)

print(f"Dataset Dimensions: {df.shape[0]:,} rows and {df.shape[1]} columns\n")
display(df.head())


In [ ]:
# Display bottom records and column list
display(df.tail(3))
print("All Dataset Columns:")
print(list(df.columns))


## 🔍 3. Schema & Data Types Validation
Inspect data types, non-null counts, and memory footprint to separate numerical and categorical features.


In [ ]:
# Dataset info and memory usage
df.info()


In [ ]:
# Categorize columns by data type
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Numerical Features ({len(num_cols)}):", num_cols)
print(f"\nCategorical Features ({len(cat_cols)}):", cat_cols)


## 🧩 4. Missing Values & Data Completeness Analysis
Identify which features contain missing values and calculate their missing percentage to determine appropriate imputation strategies.


In [ ]:
# Calculate missing value counts and percentages
missing_df = pd.DataFrame({
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage (%)': (df.isnull().sum() / len(df) * 100).round(2)
}).query('Missing_Count > 0').sort_values(by='Missing_Count', ascending=False)

if not missing_df.empty:
    print("Columns with Missing Values:")
    display(missing_df)
else:
    print("No missing values found!")


In [ ]:
# Missing values visualization
if not missing_df.empty:
    plt.figure(figsize=(8, 4))
    sns.barplot(x=missing_df.index, y=missing_df['Missing_Percentage (%)'], palette='rocket')
    plt.title('Missing Value Percentage by Column', fontsize=14, fontweight='bold')
    plt.ylabel('Missing Percentage (%)')
    plt.xlabel('Features')
    for i, v in enumerate(missing_df['Missing_Percentage (%)']):
        plt.text(i, v + 0.5, f"{v}%", ha='center', fontweight='bold')
    plt.ylim(0, max(missing_df['Missing_Percentage (%)']) + 5)
    plt.tight_layout()
    plt.show()


## 👥 5. Duplicate Records & Client ID Uniqueness
Verify whether client identifiers are unique and whether identical duplicate rows exist in the dataset.


In [ ]:
# Check total duplicate rows across all columns
duplicate_count = df.duplicated().sum()
print(f"Total fully duplicated rows: {duplicate_count}")

# Check Client ID uniqueness
if 'client_ID' in df.columns:
    unique_clients = df['client_ID'].nunique()
    total_clients = len(df)
    print(f"Total Client IDs: {total_clients:,} | Unique Client IDs: {unique_clients:,}")
    if unique_clients < total_clients:
        print(f"⚠️ Warning: Found {total_clients - unique_clients} duplicate client_ID entries!")
    else:
        print("✅ All client_ID entries are unique.")


## 📈 6. Statistical Summary & Descriptive Metrics
Examine central tendency (mean, median), dispersion (std, IQR), and extremes (min, max) for numerical and categorical variables.


In [ ]:
# Statistical summary of numerical features
print("Descriptive Statistics - Numerical Attributes:")
display(df.describe().T[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']])


In [ ]:
# Statistical summary of categorical features
print("Descriptive Statistics - Categorical Attributes:")
display(df.describe(include=['O']).T)


## 🎯 7. Target Variable Analysis: `loan_status`
Analyze the class distribution of the target variable `loan_status` (0 = Non-Default / Paid, 1 = Default) to check for class imbalance.


In [ ]:
# Target variable distribution
status_counts = df['loan_status'].value_counts()
status_pcts = df['loan_status'].value_counts(normalize=True) * 100

target_summary = pd.DataFrame({
    'Count': status_counts,
    'Percentage (%)': status_pcts.round(2)
})
target_summary.index = ['Non-Default (0)', 'Default (1)']
display(target_summary)


In [ ]:
# Visualizing target class distribution
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Donut chart
ax[0].pie(status_counts, labels=['Non-Default (0)', 'Default (1)'], autopct='%1.1f%%', 
          startangle=140, colors=['#2ecc71', '#e74c3c'], explode=(0, 0.08), 
          wedgeprops={'edgecolor': 'white', 'linewidth': 2})
ax[0].set_title('Target Distribution: Loan Status', fontsize=14, fontweight='bold')

# Countplot
sns.countplot(data=df, x='loan_status', palette=['#2ecc71', '#e74c3c'], ax=ax[1])
ax[1].set_title('Count of Defaults vs Non-Defaults', fontsize=14, fontweight='bold')
ax[1].set_xticklabels(['Non-Default (0)', 'Default (1)'])
ax[1].set_ylabel('Number of Borrowers')

plt.tight_layout()
plt.show()


## 📊 8. Univariate Analysis & Outlier Detection (Numerical)
Investigate the distributions of major numerical risk features and detect potential anomalies / extreme outliers (e.g., `person_age > 100`, extreme employment lengths, debt ratios).


In [ ]:
# Key numerical features for distribution analysis
key_num_features = ['person_age', 'person_income', 'person_emp_length', 'loan_amnt', 'loan_int_rate', 'debt_to_income_ratio']

fig, axes = plt.subplots(len(key_num_features), 2, figsize=(15, 4 * len(key_num_features)))

for i, col in enumerate(key_num_features):
    # Histogram / KDE
    sns.histplot(df[col].dropna(), kde=True, ax=axes[i, 0], color='steelblue', bins=30)
    axes[i, 0].set_title(f'{col} - Distribution & KDE', fontweight='bold')
    
    # Boxplot for outlier detection
    sns.boxplot(x=df[col].dropna(), ax=axes[i, 1], color='coral')
    axes[i, 1].set_title(f'{col} - Boxplot (Outliers)', fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
# Identify specific anomalies / domain outliers
print("Potential Anomalies Detected:")
print(f"- Borrowers with Age > 100: {len(df[df['person_age'] > 100])}")
print(f"- Borrowers with Employment Length > 60 years: {len(df[df['person_emp_length'] > 60])}")
print(f"- Max Person Income: ${df['person_income'].max():,}")
print(f"- Max Loan Amount: ${df['loan_amnt'].max():,}")


## 🏷️ 9. Univariate Analysis: Categorical Risk Factors
Explore frequency distributions across loan purpose, home ownership type, loan grade rating, and past default history.


In [ ]:
cat_analysis_cols = ['person_home_ownership', 'loan_intent', 'loan_grade', 'cb_person_default_on_file', 'education_level', 'employment_type']

fig, axes = plt.subplots(3, 2, figsize=(16, 15))
axes = axes.flatten()

for i, col in enumerate(cat_analysis_cols):
    if col in df.columns:
        order = df[col].value_counts().index
        sns.countplot(data=df, x=col, order=order, ax=axes[i], palette='viridis')
        axes[i].set_title(f'Distribution of {col}', fontweight='bold', fontsize=12)
        axes[i].tick_params(axis='x', rotation=30)
        axes[i].set_ylabel('Count')

plt.tight_layout()
plt.show()


## 🔀 10. Bivariate Analysis: Relationship with Default Risk (`loan_status`)
Examine how default rates vary across borrower categories (loan grade, loan intent, home ownership, and credit history).


In [ ]:
# Helper function to plot default rate by category
def plot_default_rate(df, cat_col, title):
    group_df = df.groupby(cat_col)['loan_status'].agg(['count', 'mean']).reset_index()
    group_df['mean'] = group_df['mean'] * 100
    group_df = group_df.sort_values(by='mean', ascending=False)
    
    plt.figure(figsize=(10, 4))
    bars = sns.barplot(data=group_df, x=cat_col, y='mean', palette='coolwarm')
    plt.title(f'Default Rate by {title}', fontsize=13, fontweight='bold')
    plt.ylabel('Default Rate (%)')
    plt.xlabel(cat_col)
    for p in bars.patches:
        bars.annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2., p.get_height()),
                      ha='center', va='center', xytext=(0, 5), textcoords='offset points', fontweight='bold')
    plt.ylim(0, max(group_df['mean']) + 10)
    plt.tight_layout()
    plt.show()

plot_default_rate(df, 'loan_grade', 'Loan Grade (A to G)')
plot_default_rate(df, 'person_home_ownership', 'Home Ownership')
plot_default_rate(df, 'loan_intent', 'Loan Intent / Purpose')
plot_default_rate(df, 'cb_person_default_on_file', 'Historical Default on File')


In [ ]:
# Bivariate numerical comparison: Loan Amount vs Income by Default Status
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df.sample(min(2000, len(df)), random_state=42), 
                x='person_income', y='loan_amnt', hue='loan_status', 
                palette={0: '#2ecc71', 1: '#e74c3c'}, alpha=0.6)
plt.title('Income vs Loan Amount (Sampled 2,000 records)', fontsize=14, fontweight='bold')
plt.xlabel('Person Annual Income ($)')
plt.ylabel('Loan Amount ($)')
plt.xlim(0, 250000)
plt.legend(title='Default Status', labels=['Non-Default (0)', 'Default (1)'])
plt.tight_layout()
plt.show()


## 🧬 11. Feature Correlation & Risk Heatmap
Calculate Pearson correlation coefficients across numerical attributes to identify collinearity and features with strongest correlation to `loan_status`.


In [ ]:
# Compute correlation matrix for numerical attributes
corr_matrix = df[num_cols].corr()

plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap='RdBu_r', 
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5, cbar_kws={"shrink": .8})
plt.title('Pearson Correlation Heatmap of Numerical Features', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Feature correlation with loan_status ranked
if 'loan_status' in corr_matrix.columns:
    target_corr = corr_matrix['loan_status'].sort_values(ascending=False).to_frame()
    target_corr.columns = ['Correlation with loan_status']
    print("Features ranked by correlation with default risk (loan_status):")
    display(target_corr)
